# Hands-On AI: RAG (Retrieval-Augmented Generation)

A deeper walkthrough of the `rag` module. By the end you will have built a small
search index over your own documents and used it to ground a model's answers.

> **Running on Colab?** Run the setup cell to point at a provider you can reach
> (see the [Notebooks & Colab guide](https://michael-borck.github.io/hands-on-ai/notebooks/)).
> **Running locally** with Ollama? You can skip it.

## What is RAG, and why?

A language model only knows what it was trained on, plus whatever you put in the
prompt. It does not know *your* notes, *your* PDFs, or anything private or recent.

RAG fixes that in three steps:

1. **Index** your documents: split them into chunks and turn each chunk into a
   vector (an embedding) that captures its meaning.
2. **Retrieve**: for a question, find the chunks whose vectors are most similar.
3. **Generate**: put those chunks in the prompt and let the model answer from them.

The model is still the brain. RAG just feeds it the right context.

## 1. Install

In [ ]:
!pip install -q hands-on-ai

## 2. Connect to a model (and an embedding model)

RAG uses **two** models: a chat model to write answers, and an *embedding* model
to turn text into vectors.

- **Local Ollama:** pull both, e.g. `ollama pull llama3` and
  `ollama pull nomic-embed-text`, then skip the cell below.
- **Colab / remote:** set a provider that offers both. The embedding model is set
  with `HANDS_ON_AI_EMBEDDING_MODEL`.

In [ ]:
import os, getpass

os.environ["HANDS_ON_AI_SERVER"] = "https://ollama.your-school.edu"   # your provider URL
os.environ["HANDS_ON_AI_MODEL"] = "llama3"                            # chat model
os.environ["HANDS_ON_AI_EMBEDDING_MODEL"] = "nomic-embed-text"        # embedding model
os.environ["HANDS_ON_AI_API_KEY"] = getpass.getpass("API key (leave blank if none): ")

## 3. Some documents to search

To *prove* the answers come from our documents and not the model's memory, we will
invent facts the model cannot possibly know.

In [ ]:
from pathlib import Path

Path("docs").mkdir(exist_ok=True)

Path("docs/zorblax_specs.txt").write_text(
    "The Zorblax 3000 is a household cleaning robot made by Polaris Robotics. "
    "It is powered by a swappable 240 watt-hour graphene battery that lasts "
    "about 5 hours per charge. The robot weighs 3.2 kilograms.")

Path("docs/zorblax_features.txt").write_text(
    "The Zorblax 3000 navigates with 12 ultrasonic sensors and a downward camera. "
    "It understands four languages: English, Spanish, Mandarin, and Hindi. "
    "It ships with a 3 year warranty.")

Path("docs/polaris_company.txt").write_text(
    "Polaris Robotics was founded in 2021 in Fremantle, Western Australia. "
    "Its motto is 'Robots that earn their keep.' The company has 47 employees.")

print("Wrote 3 documents to docs/")

## 4. Build the index

This is the part a framework would hide. Here it is in the open: read each file,
split it into chunks, embed the chunks, and save everything to one `.npz` file.

In [ ]:
from hands_on_ai.rag.utils import (
    load_text_file, chunk_text, get_embeddings, save_index_with_sources
)

chunks, sources = [], []
for path in sorted(Path("docs").glob("*.txt")):
    text = load_text_file(path)
    for chunk in chunk_text(text):
        chunks.append(chunk)
        sources.append(str(path))

print(f"Split into {len(chunks)} chunks. Embedding them...")
vectors = get_embeddings(chunks)          # one vector per chunk
save_index_with_sources(vectors, chunks, sources, "rag_demo.npz")
print(f"Saved index with shape {vectors.shape} to rag_demo.npz")

## 5. Retrieve: see what matches

`get_top_k` embeds your question, compares it to every chunk, and returns the
closest ones with a similarity score. Looking at the scores is how you *debug*
RAG: if retrieval is bad, the answer will be too.

In [ ]:
from hands_on_ai.rag.utils import get_top_k

question = "What battery does the Zorblax 3000 use, and how long does it last?"
results, scores = get_top_k(question, "rag_demo.npz", k=3, return_scores=True)

for (chunk, source), score in zip(results, scores):
    print(f"score={score:.3f}  {source}")
    print(f"   {chunk[:90]}...")
    print()

## 6. Generate a grounded answer

Put the retrieved chunks in the prompt and ask the model to answer only from them.

In [ ]:
from hands_on_ai.chat import get_response

context = "\n\n".join(chunk for chunk, _ in results)
prompt = f"""Use only the context below to answer the question.
If the answer is not in the context, say you don't know.

Context:
{context}

Question: {question}"""

print(get_response(prompt, system="You answer strictly from the provided context."))

## 7. The proof: ask without RAG

Now ask the *same* question with no context. The model has never heard of the
Zorblax 3000, so it should admit it does not know (or make something up). That gap
is exactly what RAG closes.

In [ ]:
print(get_response(question))

## Doing it from the terminal

Everything above also works as two CLI commands, no Python required:

```bash
rag index docs/                 # build the index
rag ask "What battery does the Zorblax 3000 use?" --context
```

## Where to go next

- **Concept:** [What Does the LLM Do in RAG?](https://michael-borck.github.io/hands-on-ai/rag-llm-explain/)
- **Build more:** the [RAG projects](https://michael-borck.github.io/hands-on-ai/projects/) (a Q&A CLI, a PDF chatbot, a hybrid bot)
- **Go bigger:** for real workloads you would swap the single `.npz` file for a
  vector database (Chroma, FAISS, Qdrant, pgvector). See [Our Approach](https://michael-borck.github.io/hands-on-ai/philosophy/).

The pattern never changes: retrieve the right context, then let the model answer.